# Классификация: СC50 > медианы

Цель: по набору дескрипторов предсказать, превышает ли значение IC50 медиану выборки, и сравнить несколько классификационных моделей с подбором гиперпараметров

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('/Users/yaroslavbaev/Desktop/miphi/data/chem_data_prepared.csv')

median_cc50 = df['CC50, mM'].median()
df['CC50_gt_median'] = (df['CC50, mM'] > median_cc50).astype(int)

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'CC50_gt_median'])
y = df['CC50_gt_median']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [4]:
def clf_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    return metrics

## Логистическая регрессия (база)

In [5]:
log_reg = LogisticRegression(max_iter=500, n_jobs=-1)
log_reg.fit(X_train_scaled, y_train)

y_val_pred_lr = log_reg.predict(X_val_scaled)
y_val_proba_lr = log_reg.predict_proba(X_val_scaled)[:, 1]
clf_metrics(y_val, y_val_pred_lr, y_val_proba_lr)

{'accuracy': 0.7375,
 'f1_macro': 0.7374589779653071,
 'roc_auc': np.float64(0.8070312500000001)}

## RandomForestClassifier + RandomizedSearch

In [6]:
rf_clf = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_clf_params = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}

rf_clf_search = RandomizedSearchCV(
    rf_clf,
    rf_clf_params,
    n_iter=25,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
rf_clf_search.fit(X_train, y_train)
rf_clf_search.best_params_, rf_clf_search.best_score_

Fitting 3 folds for each of 25 candidates, totalling 75 fits


({'n_estimators': 400,
  'min_samples_split': 10,
  'min_samples_leaf': 4,
  'max_features': 'sqrt',
  'max_depth': None},
 np.float64(0.826521472584736))

In [8]:
rf_cc50_best = rf_clf_search.best_estimator_
y_val_pred_rf = rf_cc50_best.predict(X_val)
y_val_proba_rf = rf_cc50_best.predict_proba(X_val)[:, 1]
clf_metrics(y_val, y_val_pred_rf, y_val_proba_rf)

{'accuracy': 0.75625,
 'f1_macro': 0.7554763117677026,
 'roc_auc': np.float64(0.8460937500000001)}

## XGBClassifier + RandomizedSearch

In [9]:
xgb_clf = XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_estimators=400,
    n_jobs=-1,
)

xgb_clf_params = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

xgb_cc50_search = RandomizedSearchCV(
    xgb_clf,
    xgb_clf_params,
    n_iter=25,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
xgb_cc50_search.fit(X_train, y_train)
xgb_cc50_search.best_params_, xgb_cc50_search.best_score_

Fitting 3 folds for each of 25 candidates, totalling 75 fits


({'subsample': 0.8,
  'max_depth': 3,
  'learning_rate': 0.01,
  'colsample_bytree': 1.0},
 np.float64(0.8361122692872026))

In [10]:
xgb_cc50_best = xgb_cc50_search.best_estimator_

y_val_pred_xgb = xgb_cc50_best.predict(X_val)
y_val_proba_xgb = xgb_cc50_best.predict_proba(X_val)[:, 1]
val_metrics_xgb = clf_metrics(y_val, y_val_pred_xgb, y_val_proba_xgb)
val_metrics_xgb

{'accuracy': 0.725,
 'f1_macro': 0.7239215686274509,
 'roc_auc': np.float64(0.8381249999999999)}

In [11]:
y_test_pred_xgb = xgb_cc50_best.predict(X_test)
y_test_proba_xgb = xgb_cc50_best.predict_proba(X_test)[:, 1]
test_metrics_xgb = clf_metrics(y_test, y_test_pred_xgb, y_test_proba_xgb)
test_metrics_xgb

{'accuracy': 0.7114427860696517,
 'f1_macro': 0.7098267622461172,
 'roc_auc': np.float64(0.8312376237623762)}

## Вывод